# Phase 3 - Data Cleaning & Validation (Real Execution)

This notebook performs the **actual, reproducible cleaning** of `data/raw/customer_churn.csv`,
based on the verified Phase-2 audit findings:

1. Load raw data
2. Inspect relevant issues
3. Apply cleaning (asserted, not silent)
4. Validate (fails loudly)
5. Compare before/after
6. Save cleaned dataset -> `data/processed/cleaned_customer_churn.csv`
7. Verify raw file integrity (SHA-256 before/after)

The raw file is **never modified**. All cleaning logic is imported from `src/data_cleaning.py`
so the notebook and the canonical script produce identical results.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import (
    RAW_PATH, CLEAN_PATH, ID_COL, TARGET, TEXT_COLUMNS, NUMERIC_COLUMNS,
    REQUIRED_COLUMNS, RANGE_CHECKS,
    sha256_of, load_raw, convert_total_charges, strip_text_columns,
    encode_target, count_iqr_outliers, validate_dataset,
)

pd.set_option("display.max_columns", 30)
print("pandas", pd.__version__)
print("Raw file  :", RAW_PATH)
print("Clean file:", CLEAN_PATH)

pandas 3.0.5
Raw file  : data\raw\customer_churn.csv
Clean file: data\processed\cleaned_customer_churn.csv


## 1. Load raw data

In [2]:
raw_hash_before = sha256_of(RAW_PATH)
print("Raw SHA-256 (before):", raw_hash_before)

df_raw = load_raw()
print(f"Original shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head()

Raw SHA-256 (before): 16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91
Original shape: 7043 rows x 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
print(df_raw.dtypes.to_string())
print("\nTotal missing (standard NaN) cells:", int(df_raw.isna().sum().sum()))

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str

Total missing (standard NaN) cells: 0


## 2. Inspect relevant issues

### 2a. TotalCharges stored as text (the only real type problem)

In [4]:
coerced = pd.to_numeric(df_raw["TotalCharges"], errors="coerce")
bad = coerced.isna()
print(f"Non-convertible TotalCharges values: {int(bad.sum())}")
print("Raw cell values (repr):", [repr(v) for v in df_raw.loc[bad, "TotalCharges"]])

problem_rows = df_raw.loc[bad, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]
display(problem_rows)

# Assert the audit conditions BEFORE any replacement happens
assert int(bad.sum()) == 11, "Expected exactly 11 non-convertible values"
assert all(isinstance(v, str) and len(v) == 1 and v.isspace() for v in df_raw.loc[bad, "TotalCharges"]), \
    "Not all non-numeric values are a single whitespace character"
assert df_raw.loc[bad, "tenure"].eq(0).all(), "Some problem rows have tenure != 0"
assert df_raw.loc[bad, "Churn"].eq("No").all(), "Some problem rows have Churn != 'No'"
print("ASSERTIONS PASSED: 11 single-whitespace cells, all tenure == 0, all Churn == 'No'")

Non-convertible TotalCharges values: 11
Raw cell values (repr): ["' '", "' '", "' '", "' '", "' '", "' '", "' '", "' '", "' '", "' '", "' '"]


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20.00,,No
5218,2923-ARZLG,0,19.70,,No
6670,4075-WKNIU,0,73.35,,No


ASSERTIONS PASSED: 11 single-whitespace cells, all tenure == 0, all Churn == 'No'


### 2b. Whitespace, category consistency, duplicates, target

In [5]:
ws_counts = {col: int(df_raw[col].ne(df_raw[col].str.strip()).sum()) for col in TEXT_COLUMNS}
print("Text columns with leading/trailing whitespace:",
      {k: v for k, v in ws_counts.items() if v} or "NONE (no categorical value needs stripping)")

collisions = {}
for col in TEXT_COLUMNS:
    groups = {}
    for u in df_raw[col].dropna().unique():
        groups.setdefault(str(u).strip().lower(), []).append(str(u))
    c = {k: v for k, v in groups.items() if len(v) > 1}
    if c:
        collisions[col] = c
print("Category normalization collisions (strip+lower):", collisions or "NONE")

print(f"Duplicate full rows : {int(df_raw.duplicated().sum())}")
print(f"Duplicate customerID: {int(df_raw[ID_COL].duplicated().sum())}")

print("Churn values:", sorted(df_raw[TARGET].unique().tolist()))
print(df_raw[TARGET].value_counts().to_string())

Text columns with leading/trailing whitespace: NONE (no categorical value needs stripping)
Category normalization collisions (strip+lower): NONE
Duplicate full rows : 0
Duplicate customerID: 0
Churn values: ['No', 'Yes']
Churn
No     5174
Yes    1869


## 3. Apply cleaning

Each step is asserted/verified before it is applied. No silent replacement.

In [6]:
# Step 1: TotalCharges conversion (asserts single-whitespace + tenure==0 + Churn=='No' on RAW values)
df, n_converted, affected_ids, invalid_rows = convert_total_charges(df_raw)
print(f"TotalCharges values converted to 0: {n_converted}")
print("Affected customerIDs:", affected_ids)
print("TotalCharges dtype after conversion:", df["TotalCharges"].dtype)

# Step 2: defensive whitespace stripping of text columns
df, ws_changes = strip_text_columns(df)
print("\nText values changed by stripping:", sum(ws_changes.values()))
print("Per-column changes:", {k: v for k, v in ws_changes.items() if v} or "none")

# Step 3: binary target encoding No -> 0, Yes -> 1
df, counts_before = encode_target(df)
print("\nChurn counts before encoding:", counts_before)
print("Churn counts after encoding :", {int(k): int(v) for k, v in df[TARGET].value_counts().items()})

TotalCharges values converted to 0: 11
Affected customerIDs: ['4472-LVYGI', '3115-CZMZD', '5709-LVOEQ', '4367-NUYAO', '1371-DWPAZ', '7644-OMVMY', '3213-VVOLG', '2520-SGTTA', '2923-ARZLG', '4075-WKNIU', '2775-SEFEE']
TotalCharges dtype after conversion: float64

Text values changed by stripping: 0
Per-column changes: none

Churn counts before encoding: {'No': 5174, 'Yes': 1869}
Churn counts after encoding : {0: 5174, 1: 1869}


In [7]:
# Duplicates: verified only - no rows removed
print(f"Duplicate full rows : {int(df.duplicated().sum())} (none removed)")
print(f"Duplicate customerID: {int(df[ID_COL].duplicated().sum())} (none removed)")

# Outliers: reported only - no values removed (audit found 0 IQR outliers)
for col in NUMERIC_COLUMNS:
    s = count_iqr_outliers(df[col])
    print(f"{col}: {s['outlier_count']} IQR outliers (bounds [{s['lower_bound']}, {s['upper_bound']}]) - retained")

Duplicate full rows : 0 (none removed)
Duplicate customerID: 0 (none removed)
tenure: 0 IQR outliers (bounds [-60.0, 124.0]) - retained
MonthlyCharges: 0 IQR outliers (bounds [-46.025, 171.375]) - retained
TotalCharges: 0 IQR outliers (bounds [-4683.525, 8868.675]) - retained


## 4. Validate

`validate_dataset` raises `ValueError` if any critical check fails (required columns,
unique IDs, duplicate rows, valid target values, numeric dtypes, missing values, ranges).

In [8]:
for msg in validate_dataset(df, "notebook-final"):
    print(msg)

[notebook-final] All critical checks passed.
tenure range OK: [0, 72] within [0, 200]
MonthlyCharges range OK: [18.25, 118.75] within [0, 1000]
TotalCharges range OK: [0.0, 8684.8] within [0, 100000]


## 5. Compare before / after

In [9]:
compare = pd.DataFrame(
    {
        "before": {
            "rows": df_raw.shape[0],
            "columns": df_raw.shape[1],
            "missing cells": int(df_raw.isna().sum().sum()),
            "duplicate rows": int(df_raw.duplicated().sum()),
            "duplicate customerIDs": int(df_raw[ID_COL].duplicated().sum()),
            "TotalCharges dtype": str(df_raw["TotalCharges"].dtype),
            "Churn values": ", ".join(sorted(df_raw[TARGET].unique())),
        },
        "after": {
            "rows": df.shape[0],
            "columns": df.shape[1],
            "missing cells": int(df.isna().sum().sum()),
            "duplicate rows": int(df.duplicated().sum()),
            "duplicate customerIDs": int(df[ID_COL].duplicated().sum()),
            "TotalCharges dtype": str(df["TotalCharges"].dtype),
            "Churn values": ", ".join(sorted(map(str, df[TARGET].unique()))),
        },
    }
)
compare

,before,after
rows,7043,7043
columns,21,21
missing cells,0,0
duplicate rows,0,0
duplicate customerIDs,0,0
TotalCharges dtype,str,float64
Churn values,"No, Yes","0, 1"


In [10]:
# The 11 affected customers: TotalCharges before vs after
before_tc = df_raw.loc[df_raw["customerID"].isin(affected_ids), ["customerID", "tenure", "TotalCharges"]]
after_tc = df.loc[df["customerID"].isin(affected_ids), ["customerID", "tenure", "TotalCharges"]]
merged = before_tc.merge(after_tc, on=["customerID", "tenure"], suffixes=("_raw", "_cleaned"))
print(f"Rows compared: {len(merged)}")
merged

Rows compared: 11


,customerID,tenure,TotalCharges_raw,TotalCharges_cleaned
0,4472-LVYGI,0,,0.0
1,3115-CZMZD,0,,0.0
2,5709-LVOEQ,0,,0.0
3,4367-NUYAO,0,,0.0
4,1371-DWPAZ,0,,0.0
5,7644-OMVMY,0,,0.0
6,3213-VVOLG,0,,0.0
7,2520-SGTTA,0,,0.0
8,2923-ARZLG,0,,0.0
9,4075-WKNIU,0,,0.0


## 6. Save cleaned dataset

In [11]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEAN_PATH, index=False)
print(f"Saved: {CLEAN_PATH}")
reloaded = pd.read_csv(CLEAN_PATH)
print(f"Reloaded shape: {reloaded.shape[0]} rows x {reloaded.shape[1]} columns")
print("Reloaded missing cells:", int(reloaded.isna().sum().sum()))

Saved: data\processed\cleaned_customer_churn.csv


Reloaded shape: 7043 rows x 21 columns
Reloaded missing cells: 0


## 7. Verify raw file integrity

In [12]:
raw_hash_after = sha256_of(RAW_PATH)
print("Raw SHA-256 (before):", raw_hash_before)
print("Raw SHA-256 (after) :", raw_hash_after)
assert raw_hash_before == raw_hash_after, "Raw dataset was modified during cleaning!"
print("PASS: data/raw/customer_churn.csv is byte-for-byte unchanged.")

Raw SHA-256 (before): 16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91
Raw SHA-256 (after) : 16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91
PASS: data/raw/customer_churn.csv is byte-for-byte unchanged.
